In [0]:
import pandas as pd
import math

In [0]:
spark.sql("use catalog proyecto_final_prueba")

DataFrame[]

In [0]:
catalog = spark.sql("select current_catalog()").first()[0]
schema = "gold"
table = "dim_ubicacion"

In [0]:
spark.sql(f"create schema if not exists {catalog}.{schema}")

DataFrame[]

In [0]:
spark.sql(f"drop table if exists {catalog}.{schema}.{table}")

DataFrame[]

In [0]:
df_silver = spark.table(f"{catalog}.silver.weather").toPandas()
df_silver


,fecha_hora_local,latitude,longitude,temperature,humidity,wind_speed,weather_code,weather_desc
0,2026-07-14 19:00:00,-6.221441,-77.88461,15.4,85,2.5,51,Lluvia ligera/moderada
1,2026-07-14 20:00:00,-6.221441,-77.88461,14.3,90,1.8,51,Lluvia ligera/moderada
2,2026-07-14 21:00:00,-6.221441,-77.88461,13.8,93,0.9,2,Nublado
3,2026-07-14 22:00:00,-6.221441,-77.88461,13.0,98,1.2,2,Nublado
4,2026-07-14 23:00:00,-6.221441,-77.88461,12.6,99,2.0,3,Nublado
...,...,...,...,...,...,...,...,...
28795,2026-01-03 14:00:00,-8.400702,-74.54935,32.3,57,3.0,0,Despejado
28796,2026-01-03 15:00:00,-8.400702,-74.54935,31.8,56,5.7,1,Nublado
28797,2026-01-03 16:00:00,-8.400702,-74.54935,29.8,68,10.9,51,Lluvia ligera/moderada
28798,2026-01-03 17:00:00,-8.400702,-74.54935,29.4,70,2.6,3,Nublado


In [0]:
dim = pd.DataFrame(df_silver[["latitude", "longitude"]]).drop_duplicates().reset_index(drop=True)
dim["id_ubicacion"] = dim.index+1
catalogo_ciudades = {
    (-6.2294, -77.8728): "Chachapoyas", (-9.5278, -77.5278): "Huaraz",
    (-13.6339, -72.8814): "Abancay", (-16.3989, -71.5350): "Arequipa",
    (-13.1588, -74.2239): "Ayacucho", (-7.1638, -78.5003): "Cajamarca",
    (-12.0566, -77.1181): "Callao", (-13.5226, -71.9673): "Cusco",
    (-12.7826, -74.9727): "Huancavelica", (-9.9306, -76.2422): "Huánuco",
    (-14.0678, -75.7286): "Ica", (-12.0651, -75.2049): "Huancayo",
    (-8.1159, -79.0300): "Trujillo", (-6.7714, -79.8409): "Chiclayo",
    (-12.0432, -77.0282): "Lima", (-3.7491, -73.2538): "Iquitos",
    (-12.5933, -69.1836): "Puerto Maldonado", (-17.1983, -70.9357): "Moquegua",
    (-10.6675, -76.2567): "Cerro de Pasco", (-5.1945, -80.6328): "Piura",
    (-15.8402, -70.0219): "Puno", (-6.0333, -76.9667): "Moyobamba",
    (-18.0146, -70.2536): "Tacna", (-3.5669, -80.4515): "Tumbes",
    (-8.3791, -74.5539): "Pucallpa"
}

# 3. Función para encontrar la ciudad matemáticamente más cercana
def obtener_ciudad_cercana(lat_api, lon_api):
    ciudad_mas_cercana = "Desconocido"
    distancia_minima = float('inf') # Infinito inicial
    
    for (lat_cat, lon_cat), nombre_ciudad in catalogo_ciudades.items():
        # Distancia euclidiana simple (teorema de Pitágoras)
        distancia = math.sqrt((lat_api - lat_cat)**2 + (lon_api - lon_cat)**2)
        
        if distancia < distancia_minima:
            distancia_minima = distancia
            ciudad_mas_cercana = nombre_ciudad
            
    return ciudad_mas_cercana

# 4. Aplicamos la nueva función a cada fila
dim['nombre_lugar'] = dim.apply(lambda row: obtener_ciudad_cercana(row['latitude'], row['longitude']), axis=1)
columns = ["id_ubicacion", "latitude", "longitude", "nombre_lugar"]
dim = dim[columns]
dim

,id_ubicacion,latitude,longitude,nombre_lugar
0,1,-6.221441,-77.884610,Chachapoyas
1,2,-9.525483,-77.545685,Huaraz
2,3,-13.673110,-72.908264,Abancay
3,4,-16.414762,-71.503330,Arequipa
4,5,-13.110720,-74.262300,Ayacucho
5,6,-7.205624,-78.502530,Cajamarca
6,7,-12.056238,-77.061980,Lima
7,8,-13.532513,-71.950530,Cusco
8,9,-12.688928,-74.918460,Huancavelica
9,10,-9.876977,-76.232510,Huánuco


In [0]:
df_spark = spark.createDataFrame(dim)
df_spark.display()

id_ubicacion,latitude,longitude,nombre_lugar
1,-6.221441,-77.88461,Chachapoyas
2,-9.525483,-77.545685,Huaraz
3,-13.67311,-72.908264,Abancay
4,-16.414762,-71.50333,Arequipa
5,-13.11072,-74.2623,Ayacucho
6,-7.2056236,-78.50253,Cajamarca
7,-12.056238,-77.06198,Lima
8,-13.532513,-71.95053,Cusco
9,-12.688928,-74.91846,Huancavelica
10,-9.876977,-76.23251,Huánuco


In [0]:
df_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.{table}")